# Problem 5: 检查点 (Checkpointing)

在本问题中，你将实现模型的保存和加载功能，
这对于长时间训练非常重要（可以中断后继续训练）。

## 5.1 模型检查点 (Model Checkpointing)

**目标**: 实现 `run_save_checkpoint` 和 `run_load_checkpoint` 函数。

检查点（checkpoint）是训练过程中模型和优化器状态的快照。
保存检查点允许我们在训练中断后恢复训练，而不需要从头开始。

**需要保存的内容**:

1. **模型状态** (`model.state_dict()`):
   - 所有的可学习参数（权重和偏置）
   - 例如：Linear 层的 weight、bias，Embedding 层的 weight 等

2. **优化器状态** (`optimizer.state_dict()`):
   - 所有参数的梯度（如果有的话）
   - Adam/AdamW 的一阶和二阶矩估计（$m$ 和 $v$）
   - 其他优化器内部状态

3. **训练迭代数** (`iteration`):
   - 当前已经完成的训练迭代次数
   - 用于恢复训练时继续正确的学习率调度

**检查点文件格式**:

使用 PyTorch 的 `.pt` 或 `.pth` 格式保存为字典：

```python
checkpoint = {
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'iteration': iteration,
}
```

**保存检查点** (`run_save_checkpoint`):

1. 获取模型的状态字典：`model_state = model.state_dict()`
2. 获取优化器的状态字典：`optimizer_state = optimizer.state_dict()`
3. 构造检查点字典
4. 使用 `torch.save()` 保存到文件

**加载检查点** (`run_load_checkpoint`):

1. 使用 `torch.load()` 从文件加载检查点
2. 加载模型状态：`model.load_state_dict(checkpoint['model'])`
3. 加载优化器状态：`optimizer.load_state_dict(checkpoint['optimizer'])`
4. 返回迭代数：`return checkpoint['iteration']`

**实现要求**:

**保存**:
- 保存模型、优化器和迭代数到指定路径
- 支持文件路径或文件对象
- 不返回任何值

**加载**:
- 从文件或文件对象加载检查点
- 恢复模型和优化器的状态
- 返回之前保存的迭代数

**对应函数**:
- `tests/adapters.py` 中的 `run_save_checkpoint(model, optimizer, iteration, out)`
- `tests/adapters.py` 中的 `run_load_checkpoint(src, model, optimizer)`

**示例用法**:

```python
# 训练循环
for iteration in range(max_iterations):
    # ... 训练代码 ...
    
    # 每 1000 次迭代保存一次检查点
    if iteration % 1000 == 0:
        run_save_checkpoint(model, optimizer, iteration, 'checkpoint.pt')

# 恢复训练
model = MyModel(...)
optimizer = AdamW(model.parameters(), ...)

# 加载检查点
start_iteration = run_load_checkpoint('checkpoint.pt', model, optimizer)

# 从上次中断的地方继续训练
for iteration in range(start_iteration + 1, max_iterations):
    # ... 继续训练 ...
```

**提示**: 
- 使用 `torch.save()` 和 `torch.load()` 函数
- `model.state_dict()` 和 `model.load_state_dict()` 是关键方法
- 注意：加载前需要先创建模型和优化器实例（它们的架构必须与保存时相同）